# Southeast Housing Market & Population Analysis

**Course:** Data Visualization (UNCG, Spring 2026)

This notebook compares population growth and the single-family housing market across five Southeastern states (NC, SC, TN, VA, WV) and six cities (Charlotte, Greensboro, Raleigh, Winston-Salem, Knoxville, Nashville).

**Data sources**
- **FRED** (Federal Reserve Economic Data): annual state and city population
- **Redfin**: monthly single-family home sales (median sale price, homes sold) at the national, state, and city level

**Notes**
- At the time of this project, Redfin data ran through February 2026. Yearly averages are limited to complete years (through 2025).
- All charts are interactive Plotly figures. Static PNG exports are in the README.

## 1. Setup

In [24]:
# Imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go

import seaborn as sns

## 2. Load Data

All raw CSVs are in the `data/` folder. Population files come from FRED; housing files come from Redfin (exported as UTF-16, tab-separated).

In [25]:
# Load city population data (FRED)
charlotte_pop_data = pd.read_csv('data/fred_charlotte_population.csv')
greensboro_pop_data = pd.read_csv('data/fred_greensboro_population.csv')
knoxville_pop_data = pd.read_csv('data/fred_knoxville_population.csv')
raleigh_pop_data = pd.read_csv('data/fred_raleigh_population.csv')
winston_pop_data = pd.read_csv('data/fred_ws_population.csv')

# Load state population data (FRED)
nc_pop_data = pd.read_csv('data/fred_nc_population.csv')
tn_pop_data = pd.read_csv('data/fred_tn_population.csv')
wv_pop_data = pd.read_csv('data/fred_wv_popupation_data.csv')
va_pop_data = pd.read_csv('data/fred_va_population_data.csv')
sc_pop_data = pd.read_csv('data/fred_sc_population_data.csv')

In [26]:
# Load housing market data (Redfin, single-family homes)
us_housing_market = pd.read_csv(
    'data/redfin_national_single_family.csv',
    encoding='utf-16',
    sep='\t',
    engine='python')  # National US market

state_housing_market = pd.read_csv(
    'data/redfin_state_housing.csv',
    encoding='utf-16',
    sep='\t',
    engine='python')  # NC, SC, TN, VA, WV

city_housing_market = pd.read_csv(
    'data/redfin_cities_housing_sinfam.csv',
    encoding='utf-16',
    sep='\t',
    engine='python')  # Charlotte, Greensboro, Raleigh, Winston-Salem, Knoxville, Nashville

## 3. Clean Housing Data

`clean_housing_df()` applies the same steps to the national, state, and city files:
- Parses `Month of Period End` into a date, year, and month
- Converts `Median Sale Price` from text (e.g. `$325K`) to a number
- Drops listing, inventory, days-on-market, and sale-to-list columns not used in this analysis

In [27]:
# Cleaning function used for all three housing files
def clean_housing_df(df):
    df = df.copy()

    # Clean column names
    df.columns = df.columns.str.strip()

    # Dates
    df['Date'] = pd.to_datetime(df['Month of Period End'], format='%B %Y')
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month_name()

    # Convert Median Sale Price text (e.g. "$325K") to a number
    df['Price_Clean'] = (
        df['Median Sale Price']
        .str.replace('$', '', regex=False)
        .str.replace('K', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
        * 1000)

    # Drop columns not used in this analysis
    cols_to_drop = [
        'Median Sale Price',
        'New Listings',
        'New Listings MoM',
        'New Listings YoY',
        'Inventory',
        'Inventory MoM',
        'Inventory YoY',
        'Days on Market',
        'Days on Market MoM',
        'Days on Market YoY',
        'Average Sale To List',
        'Average Sale To List MoM',
        'Average Sale To List YoY']

    df = df.drop(columns=cols_to_drop, errors='ignore')

    # Rename cleaned price column
    df = df.rename(columns={'Price_Clean': 'Median Sale Price'})

    # Reorder columns
    col_order = [
        'Region', 'Year', 'Month', 'Median Sale Price',
        'Median Sale Price MoM', 'Median Sale Price YoY',
        'Homes Sold', 'Homes Sold MoM', 'Homes Sold YoY', 'Month of Period End']

    return df[col_order]

city_housing_market = clean_housing_df(city_housing_market)
state_housing_market = clean_housing_df(state_housing_market)
us_housing_market = clean_housing_df(us_housing_market)

# Split city "Region" (e.g. "Charlotte, NC") into City and State
city_housing_market[['City', 'State']] = (
    city_housing_market['Region'].str.split(', ', expand=True))

In [28]:
# Save cleaned housing data
city_housing_market.to_csv('data/city_housing_cleaned.csv')
state_housing_market.to_csv('data/state_housing_cleaned.csv')
us_housing_market.to_csv('data/us_housing_cleaned.csv')

In [ ]:
# Convert date and Homes Sold columns to proper types for time-series charts
for df in [city_housing_market, state_housing_market]:
    df['Month of Period End'] = pd.to_datetime(
        df['Month of Period End'],
        format='%B %Y')

    df['Homes Sold'] = (
        df['Homes Sold']
        .astype(str)
        .str.replace(',', '', regex=False)
        .astype(float))

# Redfin data ends in Feb 2026 - Only complete years for yearly/time-series charts
LAST_FULL_YEAR = 2025

## 4. Clean & Combine Population Data

FRED reports population in thousands - Values are multiplied by 1,000. This section builds:
- `pop_df`: most recent population for each state (map and bar chart)
- `pop_combined`: full state population history
- `city_pop_combined`: full city population history
- `pop_yearly`: year-over-year population growth (%) by state

In [30]:
# Sort state population data by date
nc_pop_data = nc_pop_data.sort_values('observation_date')
tn_pop_data = tn_pop_data.sort_values('observation_date')
wv_pop_data = wv_pop_data.sort_values('observation_date')
va_pop_data = va_pop_data.sort_values('observation_date')
sc_pop_data = sc_pop_data.sort_values('observation_date')

# Most recent population for each state
pop_df = pd.DataFrame({
    'State': ['NC', 'TN', 'WV', 'VA', 'SC'],
    'Population': [
        nc_pop_data.iloc[-1]['NCPOP'] * 1000,
        tn_pop_data.iloc[-1]['TNPOP'] * 1000,
        wv_pop_data.iloc[-1]['WVPOP'] * 1000,
        va_pop_data.iloc[-1]['VAPOP'] * 1000,
        sc_pop_data.iloc[-1]['SCPOP'] * 1000]})

# Label for hover text (population in millions)
pop_df['Pop_Label'] = (pop_df['Population'] / 1_000_000).round(1).astype(str) + 'M'

In [31]:
# Combine state population histories into one DataFrame
nc = pd.DataFrame({
    'Date': pd.to_datetime(nc_pop_data['observation_date']),
    'Population': nc_pop_data['NCPOP'] * 1000,
    'State': 'NC'})

tn = pd.DataFrame({
    'Date': pd.to_datetime(tn_pop_data['observation_date']),
    'Population': tn_pop_data['TNPOP'] * 1000,
    'State': 'TN'})

sc = pd.DataFrame({
    'Date': pd.to_datetime(sc_pop_data['observation_date']),
    'Population': sc_pop_data['SCPOP'] * 1000,
    'State': 'SC'})

va = pd.DataFrame({
    'Date': pd.to_datetime(va_pop_data['observation_date']),
    'Population': va_pop_data['VAPOP'] * 1000,
    'State': 'VA'})

wv = pd.DataFrame({
    'Date': pd.to_datetime(wv_pop_data['observation_date']),
    'Population': wv_pop_data['WVPOP'] * 1000,
    'State': 'WV'})

pop_combined = pd.concat([nc, tn, sc, va, wv])

pop_combined.to_csv('data/combined_pop_data.csv')

In [32]:
# Combine city population histories into one DataFrame
charlotte = pd.DataFrame({
    'Date': pd.to_datetime(charlotte_pop_data['observation_date']),
    'Population': charlotte_pop_data.iloc[:, 1] * 1000,
    'City': 'Charlotte'})

greensboro = pd.DataFrame({
    'Date': pd.to_datetime(greensboro_pop_data['observation_date']),
    'Population': greensboro_pop_data.iloc[:, 1] * 1000,
    'City': 'Greensboro'})

knoxville = pd.DataFrame({
    'Date': pd.to_datetime(knoxville_pop_data['observation_date']),
    'Population': knoxville_pop_data.iloc[:, 1] * 1000,
    'City': 'Knoxville'})

raleigh = pd.DataFrame({
    'Date': pd.to_datetime(raleigh_pop_data['observation_date']),
    'Population': raleigh_pop_data.iloc[:, 1] * 1000,
    'City': 'Raleigh'})

winston = pd.DataFrame({
    'Date': pd.to_datetime(winston_pop_data['observation_date']),
    'Population': winston_pop_data.iloc[:, 1] * 1000,
    'City': 'Winston-Salem'})

city_pop_combined = pd.concat([
    charlotte, greensboro, knoxville, raleigh, winston])

city_pop_combined['Year'] = city_pop_combined['Date'].dt.year
city_pop_combined.to_csv('data/city_pop_combined.csv')

In [33]:
# Calculate annual population growth (%) by state
pop_combined['Year'] = pop_combined['Date'].dt.year

pop_yearly = (
    pop_combined
    .groupby(['Year', 'State'], as_index=False)['Population']
    .mean()
    .sort_values('Year'))

pop_yearly['Pop_Growth_%'] = (
    pop_yearly
    .groupby('State')['Population']
    .pct_change() * 100)

## 5. State Population

State colors are based on each state's flagship university (UNC, South Carolina, Tennessee, Virginia Tech, West Virginia) and are reused across all state charts.

In [46]:
# Map: most recent population by state
fig = px.choropleth(
    pop_df,
    locations='State',
    locationmode='USA-states',
    color='Population',
    scope='usa',
    color_continuous_scale='Viridis',
    title='State Population (2025)')

fig.update_layout(template='plotly_dark')

fig.update_traces(
    customdata=pop_df[['State', 'Pop_Label']],
    hovertemplate=
        'State: %{customdata[0]}<br>' +
        'Pop: %{customdata[1]}' +
        '<extra></extra>')

fig.show()

In [35]:
# Bar chart: state population comparison
color_map = {
    'NC': '#4B9CD3',   # UNC Blue
    'SC': '#73000A',   # Gamecocks Garnet
    'TN': '#FF8200',   # Tennessee Orange
    'VA': '#861F41',   # VT Maroon
    'WV': '#002855'}   # WVU Navy Blue

fig = px.bar(
    pop_df,
    x='State',
    y='Population',
    color='State',
    color_discrete_map=color_map,
    title='State Population Comparison')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='State',
    yaxis_title='Population',
    showlegend=False)

fig.update_traces(
    customdata=pop_df[['Pop_Label']],
    hovertemplate=
        'State: %{x}<br>' +
        'Pop: %{customdata[0]}' +
        '<extra></extra>')

fig.show()

In [36]:
# Line chart: annual population growth (%) by state, 2012 onward
fig = px.line(
    pop_yearly[pop_yearly['Year'] >= 2012],
    x='Year',
    y='Pop_Growth_%',
    color='State',
    color_discrete_map=color_map,
    title='Population Growth Rate by State (%)')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Growth (%)')
fig.update_yaxes(ticksuffix='%')
fig.show()

## 6. State Housing Market

In [47]:
# Grouped bar chart: average median home price by state, 2024 vs 2025
year_color_map = {
    '2024': '#145A32',   # dark green
    '2025': '#28A745'}   # regular green

df_bar = (state_housing_market[state_housing_market['Year'].isin([2024, 2025])]
          .groupby(['Year', 'Region'], as_index=False)['Median Sale Price'].mean())

# Year as text so Plotly treats it as a category, not a number
df_bar['Year'] = df_bar['Year'].astype(str)

fig = px.bar(
    df_bar,
    x='Region',
    y='Median Sale Price',
    color='Year',
    color_discrete_map=year_color_map,
    barmode='group',
    title='Median Home Price by State (2024 vs 2025)')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='State',
    yaxis_title='Median Home Price ($)')

fig.update_yaxes(tickformat='$,')

fig.show()

In [38]:
# Line chart: average median home price per year by state (complete years only)
state_name_color_map = {
    'North Carolina': '#4B9CD3',
    'South Carolina': '#73000A',
    'Tennessee': '#FF8200',
    'Virginia': '#861F41',
    'West Virginia': '#002855'}

fig = px.line(
    state_housing_market[state_housing_market['Year'] <= LAST_FULL_YEAR]
    .groupby(['Year', 'Region'], as_index=False)['Median Sale Price']
    .mean()
    .sort_values('Year'),

    x='Year',
    y='Median Sale Price',
    color='Region',
    color_discrete_map=state_name_color_map,
    title='Average Median Home Price per Year by State')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Median Home Price ($)')

fig.update_yaxes(tickformat='$,')
fig.show()

In [39]:
# Line chart: statewide homes sold, NC vs TN (2023–2025)
nc_tn_homes_sold = state_housing_market[
    (state_housing_market['Region'].isin(['North Carolina', 'Tennessee'])) &
    (state_housing_market['Month of Period End'].dt.year >= 2023) &
    (state_housing_market['Month of Period End'].dt.year <= LAST_FULL_YEAR)]

fig = px.line(
    nc_tn_homes_sold,
    x='Month of Period End',
    y='Homes Sold',
    color='Region',
    color_discrete_map=state_name_color_map,
    title='Homes Sold Over Time: NC vs TN (2023–2025)')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Homes Sold',
    legend_title='State')

fig.update_yaxes(tickformat=',')

fig.show()

## 7. City Population

City colors are based on each city's local university (Charlotte 49ers, UNCG, Wake Forest, UNC, Tennessee).

In [40]:
# Line chart: population over time by city
city_color_map = {
    'Knoxville': '#FF8200',       # Tennessee Orange
    'Winston-Salem': '#C99700',   # Wake Forest Gold
    'Charlotte': '#046A38',       # Charlotte 49ers Green
    'Greensboro': '#0F2044',      # UNCG Blue
    'Raleigh': '#4B9CD3'}         # UNC Blue

city_pop_yearly = (
    city_pop_combined
    .groupby(['Year', 'City'], as_index=False)['Population']
    .mean()
    .sort_values('Year'))

city_pop_yearly['Population_M'] = city_pop_yearly['Population'] / 1_000_000

fig = px.line(
    city_pop_yearly[city_pop_yearly['Year'] >= 2012],
    x='Year',
    y='Population_M',
    color='City',
    color_discrete_map=city_color_map,
    title='Population Over Time by City')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Population (Millions)',
    legend_title='City')

fig.show()

In [48]:
# Bar chart: most recent population by city
latest_city_pop = (
    city_pop_combined
    .sort_values('Date')
    .groupby('City')
    .tail(1))

latest_city_pop['Population_M'] = latest_city_pop['Population'] / 1_000_000

fig = px.bar(
    latest_city_pop,
    x='City',
    y='Population_M',
    color='City',
    color_discrete_map=city_color_map,
    title='Latest (2026) Population by City')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='City',
    yaxis_title='Population (Millions)',
    showlegend=False)

fig.update_yaxes(tickformat=',.1f')

fig.show()

In [42]:
# Table: most recent population by city
table_city_pop = latest_city_pop[['City', 'Population_M']].copy()
table_city_pop['Population_M'] = table_city_pop['Population_M'].round(1).astype(str) + 'M'
table_city_pop.columns = ['City', 'Population (Millions)']

fig = go.Figure(data=[go.Table(
    header=dict(
        values=list(table_city_pop.columns),
        fill_color='#2c2c2c',          # dark gray
        font=dict(color='white', size=12),
        align='left'),

    cells=dict(
        values=[table_city_pop[col] for col in table_city_pop.columns],
        fill_color='#1e1e1e',          # slightly lighter dark gray
        font=dict(color='white', size=11),
        align='left'))])

fig.update_layout(
    title='Latest Population by City',
    template='plotly_dark')

fig.show()

## 8. City Housing Market

Nashville is included in the housing data but not the population data. Knoxville is gray in these charts so Tennessee orange isn't shared between two cities.

In [49]:
# Grouped bar chart: average median home price by city, 2024 vs 2025
year_color_map = {
    '2024': '#1f3a5f',   # dark blue
    '2025': '#1f77b4'}   # regular blue

df_bar_city = (
    city_housing_market[
        city_housing_market['Year'].isin([2024, 2025])]
    .groupby(['Year', 'City'], as_index=False)['Median Sale Price']
    .mean())

# Year as text so Plotly treats it as a category, not a number
df_bar_city['Year'] = df_bar_city['Year'].astype(str)

fig = px.bar(
    df_bar_city,
    x='City',
    y='Median Sale Price',
    color='Year',
    color_discrete_map=year_color_map,
    barmode='group',
    title='Median Home Price by City (2024 vs 2025)')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='City',
    yaxis_title='Median Home Price ($)')

fig.update_yaxes(tickformat='$,')
fig.show()

In [44]:
# Line chart: monthly homes sold by city, 2023–2025
city_color_map = {
    'Knoxville': 'Gray',
    'Winston-Salem': '#C99700',
    'Charlotte': '#046A38',
    'Greensboro': '#0F2044',
    'Raleigh': '#4B9CD3',
    'Nashville': 'Purple'}

fig = px.line(
    city_housing_market[
        (city_housing_market['Month of Period End'].dt.year >= 2023) &
        (city_housing_market['Month of Period End'].dt.year <= LAST_FULL_YEAR)],
    x='Month of Period End',
    y='Homes Sold',
    color='City',
    color_discrete_map=city_color_map,
    title='Homes Sold Over Time by City (2023–2025)')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Homes Sold',
    legend_title='City')

fig.update_yaxes(tickformat=',')

fig.show()

In [45]:
# Line chart: average median home price per year by city (complete years only)
fig = px.line(
    city_housing_market[city_housing_market['Year'] <= LAST_FULL_YEAR]
    .groupby(['Year', 'City'], as_index=False)['Median Sale Price']
    .mean()
    .sort_values('Year'),
    x='Year',
    y='Median Sale Price',
    color='City',
    color_discrete_map=city_color_map,
    title='Average Median Home Price per Year by City')

fig.update_layout(
    template='plotly_dark',
    xaxis_title='Year',
    yaxis_title='Median Home Price ($)',
    legend_title='City')

fig.update_yaxes(tickformat='$,')

fig.show()